# Day 07 — Mock Interview Practice

## Difficulty Level: BASIC → INTERMEDIATE (Week 1, Day 7 — final foundations day)

### Today's Focus
Last day of Week 1. Consolidate fundamentals across 8 core categories with 18 interview questions, then we step up to INTERMEDIATE→ADVANCED in Week 2. Each day builds progressively on previous topics — no question is repeated from days 1–6.

### Categories
| # | Category | Questions | Difficulty |
|---|----------|-----------|------------|
| 1 | SQL Advanced | 2 | BASIC, INTERMEDIATE |
| 2 | Python — Loop/Function/List/Dict | 2 | BASIC, INTERMEDIATE |
| 3 | PySpark Coding | 2 | BASIC, INTERMEDIATE |
| 4 | Spark Optimization & Deep Spark | 2 | BASIC, INTERMEDIATE |
| 5 | ETL — Failures, Late Data, Schema Handling | 3 | BASIC, INTERMEDIATE |
| 6 | Data Modeling | 2 | BASIC, INTERMEDIATE |
| 7 | Snowflake | 2 | BASIC, INTERMEDIATE |
| 8 | Monitoring, Logging & Orchestration | 3 | BASIC, INTERMEDIATE |

**Total: 18 Questions | 8 Categories**

---

## Q1: SQL Advanced — NTILE Bucketing into Quartiles [BASIC]

### Problem
You have a `customer_spend` table with `customer_id` and `total_spend`. Bucket customers into **4 quartiles** based on `total_spend` (highest spend = quartile 1). Also return the count of customers and average spend in each quartile.

### Sample Data
| customer_id | total_spend |
|-------------|------------|
| 1 | 5000 |
| 2 | 1500 |
| 3 | 8200 |
| 4 | 300 |
| 5 | 4100 |
| 6 | 9900 |
| 7 | 2200 |
| 8 | 6700 |

### Expected Output (per customer + quartile summary)
Each customer tagged with `quartile` 1–4; summary shows avg spend per quartile.

### Key Concept
`NTILE(n)` divides an ordered partition into `n` buckets of as-equal-as-possible size. Useful for percentile/quartile segmentation, A/B cohorts, and revenue banding.

In [0]:
from pyspark.sql import Row

data = [
    Row(customer_id=1, total_spend=5000),
    Row(customer_id=2, total_spend=1500),
    Row(customer_id=3, total_spend=8200),
    Row(customer_id=4, total_spend=300),
    Row(customer_id=5, total_spend=4100),
    Row(customer_id=6, total_spend=9900),
    Row(customer_id=7, total_spend=2200),
    Row(customer_id=8, total_spend=6700),
]

df = spark.createDataFrame(data)
df.createOrReplaceTempView("customer_spend")

# Tag each customer with a quartile (highest spend => quartile 1)
spark.sql("""
    SELECT 
        customer_id,
        total_spend,
        NTILE(4) OVER (ORDER BY total_spend DESC) AS quartile
    FROM customer_spend
    ORDER BY quartile, total_spend DESC
""").show()

# Summary: count + avg spend per quartile
spark.sql("""
    WITH tagged AS (
        SELECT customer_id, total_spend,
               NTILE(4) OVER (ORDER BY total_spend DESC) AS quartile
        FROM customer_spend
    )
    SELECT 
        quartile,
        COUNT(*) AS customer_count,
        ROUND(AVG(total_spend), 2) AS avg_spend,
        MIN(total_spend) AS min_spend,
        MAX(total_spend) AS max_spend
    FROM tagged
    GROUP BY quartile
    ORDER BY quartile
""").show()

## Q2: SQL Advanced — Rolling 7-Day Moving Average with RANGE BETWEEN [INTERMEDIATE]

### Problem
Given a `daily_revenue` table (`date`, `revenue`), compute a **rolling 7-day moving average** of revenue — the average of all rows whose `date` falls within the 6 days before and the current day. Distinguish `ROWS BETWEEN` (row-count frame) from `RANGE BETWEEN` (value-range frame) and explain when each is correct.

### Sample Data
| date | revenue |
|------|---------|
| 2026-01-01 | 100 |
| 2026-01-02 | 120 |
| 2026-01-03 | 90 |
| 2026-01-05 | 200 |
| 2026-01-06 | 150 |
| 2026-01-07 | 180 |
| 2026-01-09 | 210 |

Note: some dates are missing (Jan 4, Jan 8). A `ROWS BETWEEN 6 PRECEDING AND CURRENT ROW` would be **wrong** here because it counts rows, not calendar days.

### Expected Output
Each date gets `ma7` = average revenue over the 7-day calendar window ending at that date.

### Key Concept
- `ROWS BETWEEN n PRECEDING` → fixed number of *rows*.
- `RANGE BETWEEN INTERVAL n DAYS PRECEDING AND CURRENT ROW` → fixed *calendar window*, robust to gaps. Use RANGE for time-series rolling windows.

In [0]:
from pyspark.sql import Row
from pyspark.sql import functions as F

data = [
    Row(date="2026-01-01", revenue=100),
    Row(date="2026-01-02", revenue=120),
    Row(date="2026-01-03", revenue=90),
    Row(date="2026-01-05", revenue=200),
    Row(date="2026-01-06", revenue=150),
    Row(date="2026-01-07", revenue=180),
    Row(date="2026-01-09", revenue=210),
]
df = spark.createDataFrame(data)
df = df.withColumn("date", F.to_date("date"))
df.createOrReplaceTempView("daily_revenue")

spark.sql("""
    SELECT 
        date,
        revenue,
        ROUND(
            AVG(revenue) OVER (
                ORDER BY date
                RANGE BETWEEN INTERVAL 6 DAYS PRECEDING AND CURRENT ROW
            ), 2
        ) AS ma7
    FROM daily_revenue
    ORDER BY date
""").show()

# Contrast: ROWS BETWEEN counts rows, NOT calendar days — wrong when dates are missing
spark.sql("""
    SELECT 
        date,
        revenue,
        ROUND(
            AVG(revenue) OVER (
                ORDER BY date
                ROWS BETWEEN 6 PRECEDING AND CURRENT ROW
            ), 2
        ) AS ma7_rows
    FROM daily_revenue
    ORDER BY date
""").show()

## Q3: Python — Two Sum [BASIC]

### Problem
Given a list of integers `nums` and an integer `target`, return the **indices** of the two numbers that add up to `target`. Assume exactly one solution exists and you may not use the same element twice.

### Example
```
nums = [2, 7, 11, 15], target = 9  ->  [0, 1]
nums = [3, 2, 4],      target = 6  ->  [1, 2]
```

### Key Concept
A hash map (dict) gives O(n) time vs the brute-force O(n²). Look up `target - num` as you iterate.

In [0]:
def two_sum(nums, target):
    seen = {}  # value -> index
    for i, num in enumerate(nums):
        complement = target - num
        if complement in seen:
            return [seen[complement], i]
        seen[num] = i
    return []  # no solution

# Test cases
assert two_sum([2, 7, 11, 15], 9) == [0, 1]
assert two_sum([3, 2, 4], 6) == [1, 2]
assert two_sum([3, 3], 6) == [0, 1]
assert two_sum([1, 5, 8, 3], 13) == [1, 2]  # 5 (idx 1) + 8 (idx 2) = 13
print("All Two Sum tests passed!")

## Q4: Python — Next Greater Element (Monotonic Stack) [INTERMEDIATE]

### Problem
Given a list of integers, for each element return the **next greater element** to its right. If none exists, return `-1`.

### Example
```
[4, 5, 2, 10, 8]  ->  [5, 10, 10, -1, -1]
[1, 3, 2, 4]      ->  [3, 4, 4, -1]
```

### Key Concept
A **monotonic decreasing stack** solves this in O(n). Traverse left to right; while the stack top is less than the current value, current is its next greater. This pattern generalises to stock-span, daily temperatures, and largest-rectangle-in-histogram.

In [0]:
def next_greater_elements(nums):
    n = len(nums)
    result = [-1] * n
    stack = []  # indices with decreasing values
    for i in range(n):
        while stack and nums[stack[-1]] < nums[i]:
            idx = stack.pop()
            result[idx] = nums[i]
        stack.append(i)
    # remaining stack indices keep -1 (no greater element)
    return result

# Test cases
assert next_greater_elements([4, 5, 2, 10, 8]) == [5, 10, 10, -1, -1]
assert next_greater_elements([1, 3, 2, 4]) == [3, 4, 4, -1]
assert next_greater_elements([5, 4, 3, 2, 1]) == [-1, -1, -1, -1, -1]
assert next_greater_elements([1, 2, 3, 4]) == [2, 3, 4, -1]
print("All Next Greater Element tests passed!")

## Q5: PySpark Coding — Conditional Columns with `when/otherwise` [BASIC]

### Problem
You have an `employees` DataFrame (`emp_id`, `name`, `salary`, `dept`). Add a `grade` column using `when/otherwise`:
- salary >= 9000  -> 'A'
- salary >= 6000  -> 'B'
- salary >= 3000  -> 'C'
- otherwise       -> 'D'

Also add a `bonus` column = 10% of salary for grade A/B, 5% otherwise, using the derived grade.

### Key Concept
`F.when(...).when(...).otherwise(...)` is the PySpark equivalent of SQL `CASE WHEN`. Chain conditions in order; first match wins.

In [0]:
from pyspark.sql import Row
from pyspark.sql import functions as F

data = [
    Row(emp_id=1, name="Alice", salary=9500, dept="Eng"),
    Row(emp_id=2, name="Bob", salary=6200, dept="Eng"),
    Row(emp_id=3, name="Carol", salary=3500, dept="Sales"),
    Row(emp_id=4, name="Dave", salary=2500, dept="Sales"),
    Row(emp_id=5, name="Eve", salary=12000, dept="Eng"),
]
df = spark.createDataFrame(data)

df = df.withColumn(
    "grade",
    F.when(df.salary >= 9000, "A")
     .when(df.salary >= 6000, "B")
     .when(df.salary >= 3000, "C")
     .otherwise("D")
)

df = df.withColumn(
    "bonus",
    F.when(df.grade.isin("A", "B"), F.round(df.salary * 0.10, 2))
     .otherwise(F.round(df.salary * 0.05, 2))
)

df.orderBy("emp_id").show()

## Q6: PySpark Coding — `collect_set` / `collect_list` Aggregation [INTERMEDIATE]

### Problem
Given an `orders` DataFrame (`customer_id`, `product`), produce one row per customer with:
1. `products_bought` — ordered list of all products (duplicates allowed) -> `collect_list`
2. `unique_products` — de-duplicated set of products -> `collect_set`
3. `num_unique` — count of unique products

### Key Concept
- `collect_list` keeps duplicates & order is non-deterministic.
- `collect_set` de-duplicates (it's a set). Both produce arrays — combine with `array_sort`, `size`, `array_distinct` for further shaping.

In [0]:
from pyspark.sql import Row
from pyspark.sql import functions as F

data = [
    Row(customer_id=1, product="laptop"),
    Row(customer_id=1, product="mouse"),
    Row(customer_id=1, product="laptop"),
    Row(customer_id=2, product="keyboard"),
    Row(customer_id=2, product="monitor"),
    Row(customer_id=2, product="keyboard"),
    Row(customer_id=3, product="usb-c cable"),
]
df = spark.createDataFrame(data)

agg = (df.groupBy("customer_id")
         .agg(
             F.collect_list("product").alias("products_bought"),
             F.collect_set("product").alias("unique_products"),
             F.countDistinct("product").alias("num_unique"),
         )
         .orderBy("customer_id"))

agg.show(truncate=False)

## Q7: Spark Optimization — `repartition` vs `coalesce` [BASIC]

### Problem
You have a DataFrame with 200 partitions and need to write it to storage efficiently.
1. Reduce to 8 partitions before writing — which method?
2. Increase parallelism for a shuffle-heavy operation to 200 partitions — which method?
Explain the difference: shuffle cost, full vs partial shuffle, when each is appropriate, and the `coalesce` "no-shuffle" caveat (it can create uneven partitions).

### Key Concept
- `coalesce(n)` → **decreases** partitions, minimises shuffle (merges existing partitions). Use before writing small files. Can produce uneven partition sizes.
- `repartition(n)` → **increases or decreases**, always does a **full shuffle**, produces evenly-sized partitions. Use before a wide transformation or when data is skewed.
- `repartition(n, col)` → partition by key (range/hash) for downstream joins.

In [0]:
from pyspark.sql import functions as F

df = spark.range(0, 1000000).toDF("id")

def num_parts(d):
    # DataFrame-friendly partition count (no RDD on serverless)
    return d.withColumn("pid", F.spark_partition_id()).select("pid").distinct().count()

print("Default partitions:", num_parts(df))

# coalesce: reduce to 8 — minimal shuffle, good before writing
df_small = df.coalesce(8)
print("After coalesce(8):", num_parts(df_small))

# repartition: full shuffle, evenly sized partitions
df_even = df.repartition(16)
print("After repartition(16):", num_parts(df_even))

# repartition by key for downstream joins
df_keyed = df.repartition(8, "id")
print("After repartition(8, 'id'):", num_parts(df_keyed))

# Inspect partition sizes (rows per partition) for coalesce vs repartition
print("\ncoalesce(8) partition sizes:",
      df_small.groupBy(F.spark_partition_id().alias("pid")).count().orderBy("pid").collect())
print("repartition(16) partition sizes (should be ~even):",
      df_even.groupBy(F.spark_partition_id().alias("pid")).count().orderBy("pid").collect())

## Q8: Spark Optimization — `spark.sql.shuffle.partitions` & Partition Tuning [INTERMEDIATE]

### Problem
A job reads a 50 GB Delta table and runs `groupBy` then a join, but it's slow with 200 default shuffle partitions (too many tiny tasks) on an 8-core cluster. How do you tune the number of shuffle partitions? Explain the formula, AQE's role (`spark.sql.adaptive.coalescePartitions.enabled`), and when to set the value manually vs let AQE coalesce.

### Key Concept
- Default `spark.sql.shuffle.partitions = 200` — too high for small/medium data, too low for huge data.
- Rule of thumb: target partition size ~128 MB. `partitions = max(1, total_shuffle_bytes / 128MB)`.
- **AQE** (Adaptive Query Execution) can coalesce small shuffle partitions at runtime: `spark.sql.adaptive.enabled=true`, `spark.sql.adaptive.coalescePartitions.enabled=true`.
- Set manually only when you know the data size and AQE is off/insufficient.

In [0]:
# --- Config-driven tuning (explanatory; some AQE configs are managed on serverless) ---
def safe_set(key, value):
    try:
        spark.conf.set(key, value)
        return True
    except Exception as e:
        print(f"  (managed, not settable here: {key} — {type(e).__name__})")
        return False

def safe_get(key, default="<managed>"):
    try:
        return spark.conf.get(key)
    except Exception:
        return default

safe_set("spark.sql.shuffle.partitions", "64")          # manual tuning (works)
safe_set("spark.sql.adaptive.enabled", "true")            # AQE — often managed on serverless
safe_set("spark.sql.adaptive.coalescePartitions.enabled", "true")
safe_set("spark.sql.adaptive.coalescePartitions.minPartitionSize", "128MB")

print("shuffle.partitions =", safe_get("spark.sql.shuffle.partitions"))
print("AQE enabled =", safe_get("spark.sql.adaptive.enabled", default="<managed: on by default>"))
print("coalescePartitions =", safe_get("spark.sql.adaptive.coalescePartitions.enabled", default="<managed>"))

# Demo: a groupBy that benefits from fewer shuffle partitions
from pyspark.sql import Row
rows = [(i % 10, i) for i in range(100000)]
df = spark.createDataFrame(rows, ["grp", "val"])

agg = df.groupBy("grp").agg(F.sum("val").alias("total"), F.count("*").alias("cnt")).orderBy("grp")
agg.show(5)
print("Shuffle partitions target =", safe_get("spark.sql.shuffle.partitions"))
print("On serverless AQE is enabled by default and managed; set shuffle.partitions manually only when needed.")

## Q9: ETL — Delta Lake `MERGE` / Upsert (SCD Type 1 basics) [BASIC]

### Problem
You have a target `customers` Delta table and a daily `customers_updates` source. Upsert: when `customer_id` matches, **overwrite** all columns with the new values (SCD Type 1). When no match, insert. Write the `MERGE INTO` statement and explain the SCD Type 1 semantics (no history kept).

### Key Concept
`MERGE INTO` is the Delta upsert primitive. SCD Type 1 = in-place overwrite, no history. Use `whenMatched().updateAll()` + `whenNotMatched().insertAll()`. `mergeSchema` lets new columns auto-evolve.

In [0]:
from pyspark.sql.types import StructType, StructField, LongType, StringType, TimestampType
from pyspark.sql import functions as F
import datetime as dt
from delta.tables import DeltaTable

# Target table (create fresh)
spark.sql("DROP TABLE IF EXISTS tmp_customers_07")
spark.sql("DROP TABLE IF EXISTS tmp_customers_updates_07")

# Schema matches the source exactly (long + timestamp) to avoid MERGE type mismatches
schema = StructType([
    StructField("customer_id", LongType(), True),
    StructField("name", StringType(), True),
    StructField("city", StringType(), True),
    StructField("updated_at", TimestampType(), True),
])

spark.sql("""
    CREATE TABLE tmp_customers_07 (customer_id BIGINT, name STRING, city STRING, updated_at TIMESTAMP)
    USING DELTA
""")

def ts(s):
    return dt.datetime.strptime(s, "%Y-%m-%d %H:%M:%S")

base = [
    (1, "Alice", "NYC", ts("2026-01-01 00:00:00")),
    (2, "Bob",   "LA",  ts("2026-01-01 00:00:00")),
]
spark.createDataFrame(base, schema).write.mode("overwrite").saveAsTable("tmp_customers_07")

# Daily update: customer 1 moved city, customer 3 is new
updates = [
    (1, "Alice", "Boston", ts("2026-01-02 00:00:00")),
    (3, "Carol", "SF",     ts("2026-01-02 00:00:00")),
]
spark.createDataFrame(updates, schema).write.mode("overwrite").saveAsTable("tmp_customers_updates_07")

# MERGE: SCD Type 1 — overwrite on match, insert on no match
spark.sql("""
    MERGE INTO tmp_customers_07 AS t
    USING tmp_customers_updates_07 AS s
    ON t.customer_id = s.customer_id
    WHEN MATCHED THEN UPDATE SET *
    WHEN NOT MATCHED THEN INSERT *
""")

spark.sql("SELECT * FROM tmp_customers_07 ORDER BY customer_id").show()
# Customer 1 city is now Boston (history lost -> SCD Type 1); customer 3 inserted.

## Q10: ETL — Auto Loader Schema Evolution (`mergeSchema`, `schemaHints`) [INTERMEDIATE]

### Problem
You ingest JSON files from a landing directory with `cloudFiles`. The schema **changes over time** — new columns appear (e.g. `discount` added later), and some fields are occasionally null/cast-ambiguous. Configure Auto Loader to:
1. Evolve the schema automatically (`mergeSchema`).
2. Provide a schema hint so `order_id` stays `long` and `amount` stays `decimal(10,2)`.
3. Rescue malformed records into a `_rescued_data` column.

### Key Concept
Auto Loader (`cloudFiles` format) streams new files incrementally. `cloudFiles.schemaLocation` persists schema. `mergeSchema=true` evolves it. `schemaHints` fixes ambiguous types. `rescuedDataColumn` quarantines parse failures.

In [0]:
# Auto Loader config (explanatory; requires a real cloud path + schema location to run)
auto_loader_opts = {
    "cloudFiles.format": "json",
    "cloudFiles.schemaLocation": "/tmp/schema_07/orders_schema",
    "cloudFiles.mergeSchema": "true",
    "cloudFiles.schemaHints": "order_id long, amount decimal(10,2), discount double",
    "rescuedDataColumn": "_rescued_data",
}

for k, v in auto_loader_opts.items():
    print(f"{k} = {v}")

# Sketch of the streaming read (not started here — needs a live landing path):
# stream_df = (spark.readStream.format("cloudFiles")
#              .options(**auto_loader_opts)
#              .load("/mnt/landing/orders"))

print("\nSchema evolution pattern:")
print("  - New JSON column -> mergeSchema adds it automatically.")
print("  - schemaHints fixes types before inference goes wrong.")
print("  - _rescued_data captures rows that don't fit the evolved schema.")

## Q11: ETL — Structured Streaming `foreachBatch` with Idempotent Sink [INTERMEDIATE]

### Problem
You stream aggregated data to a Delta table in micro-batches using `foreachBatch`. After a restart, some batches replayed, causing duplicate rows. Design an **idempotent** sink: each batch upserts on a stable key so re-runs don't duplicate. Explain checkpointing + exactly-once-ish guarantees with `foreachBatch`.

### Key Concept
- `foreachBatch` lets you apply arbitrary batch logic (Delta MERGE) per micro-batch.
- Idempotency = re-running the same batch yields the same result. MERGE on `(event_id, batch_id)` makes replays safe.
- `checkpointLocation` stores offsets/commits so the stream resumes after restart without reprocessing committed batches.
- Structured Streaming + Delta gives **exactly-once** semantics when the sink is idempotent and checkpointed.

In [0]:
from pyspark.sql import functions as F

# Idempotent upsert function used inside foreachBatch
def upsert_batch(micro_batch_df, batch_id):
    # MERGE on a stable key so replaying this batch is idempotent
    micro_batch_df.createOrReplaceTempView("_updates")
    spark.sql(f"""
        MERGE INTO tmp_stream_target_07 AS t
        USING _updates AS s
        ON t.event_id = s.event_id
        WHEN MATCHED THEN UPDATE SET *
        WHEN NOT MATCHED THEN INSERT *
    """)

# Explanatory sketch (needs a live stream + checkpoint + target table):
# (spark.readStream.format("delta").load("/tmp/events")
#      .withColumn("event_id", F.col("id"))
#      .writeStream
#      .foreachBatch(upsert_batch)
#      .option("checkpointLocation", "/tmp/cp_07/events")
#      .outputMode("append")
#      .start())

print("Idempotent foreachBatch pattern:")
print("  1. Stable key (event_id) -> MERGE makes replays safe.")
print("  2. checkpointLocation stores offsets -> resume without re-reading committed data.")
print("  3. Delta target + idempotent MERGE => exactly-once effect.")

## Q12: Data Modeling — Kimball vs Inmon [BASIC]

### Problem
Explain the two dominant data-warehouse design philosophies and when to use each.

### Key Concept
| Aspect | Kimball (bottom-up) | Inmon (top-down) |
|---|---|---|
| Approach | Build dimensional marts first, then integrate via **conformed dimensions** / bus architecture | Build a normalised **3NF enterprise data warehouse** first, then derive dimensional marts |
| Structure | Star/snowflake schemas | Normalised (3NF) EDW + marts |
| Time to value | Fast (departmental) | Slower (enterprise first) |
| Best for | BI / reporting / self-serve analytics | Single source of truth, complex integration |

**Modern take:** the **medallion architecture** (bronze/silver/gold) often combines both — silver = Inmon-style normalised EDW, gold = Kimball-style star schemas.

In [0]:
# Illustrative comparison (no execution needed — conceptual)
print("Kimball (bottom-up):")
print("  - Star schemas per business process (sales, inventory).")
print("  - Conformed dimensions shared across marts (dim_date, dim_product).")
print("  - Fast delivery, BI-friendly.\n")

print("Inmon (top-down):")
print("  - 3NF enterprise data warehouse first.")
print("  - Dimensional marts derived from EDW.")
print("  - Single source of truth, slower to deliver.\n")

print("Modern: Medallion = Inmon (bronze/silver normalized) + Kimball (gold star schemas).")

## Q13: Data Modeling — Conformed & Role-Playing Dimensions [INTERMEDIATE]

### Problem
A sales fact has THREE date columns: `order_date`, `ship_date`, `return_date`. A single `dim_date` must serve all three. Explain how to model this with **role-playing dimensions** (multiple aliases of one conformed dimension). Also define **conformed dimension** and **degenerate dimension**.

### Key Concept
- **Conformed dimension**: a dimension shared across fact tables with consistent meaning/keys (e.g. `dim_date`, `dim_product`).
- **Role-playing dimension**: the same physical dim joined to a fact multiple times under different aliases (`dim_date AS order_date_dim`, `dim_date AS ship_date_dim`). In SQL/Spark views, use aliased views or a single `dim_date` joined on different FKs.
- **Degenerate dimension**: a dimension key stored in the fact with no separate dim table (e.g. `order_number`, `invoice_number` used as a grouping header).

In [0]:
from pyspark.sql import Row
from pyspark.sql import functions as F

date_dim = spark.createDataFrame([
    Row(date_sk=20260101, date="2026-01-01", month="Jan", weekday="Thu"),
    Row(date_sk=20260103, date="2026-01-03", month="Jan", weekday="Sat"),
    Row(date_sk=20260105, date="2026-01-05", month="Jan", weekday="Mon"),
])
date_dim.createOrReplaceTempView("dim_date")

sales = spark.createDataFrame([
    Row(order_id=1, order_date_sk=20260101, ship_date_sk=20260103, return_date_sk=20260105, amount=100),
    Row(order_id=2, order_date_sk=20260103, ship_date_sk=20260105, return_date_sk=None, amount=250),
])
sales.createOrReplaceTempView("fact_sales")

# Role-playing: join dim_date THREE times under different aliases
spark.sql("""
    SELECT 
        f.order_id,
        f.amount,
        od.date AS order_date,  od.month AS order_month,
        sd.date AS ship_date,   sd.month AS ship_month,
        rd.date AS return_date
    FROM fact_sales f
    LEFT JOIN dim_date od ON f.order_date_sk = od.date_sk
    LEFT JOIN dim_date sd ON f.ship_date_sk  = sd.date_sk
    LEFT JOIN dim_date rd ON f.return_date_sk = rd.date_sk
    ORDER BY f.order_id
""").show()

print("order_id is a DEGENERATE dimension (grouping key, no separate dim table).")

## Q14: Snowflake — Architecture & Virtual Warehouses [BASIC]

### Problem
Describe Snowflake's three-layer architecture and how compute scales independently of storage. Explain **virtual warehouses**, auto-suspend, auto-resume, and multi-cluster warehouses.

### Key Concept
| Layer | Responsibility |
|---|---|
| **Cloud Services** (global) | Metadata, access control, optimizer, transaction mgmt, security |
| **Query Processing** (virtual warehouses) | Compute — executes queries; independent, elastic |
| **Storage** (cloud object store) | Micro-partitioned, compressed table data; charged by TB/month |

- **Virtual warehouse**: a cluster of compute resources (T-shirt sizing: X-Small…6X-Large). Scales compute independently of storage.
- **Auto-suspend**: warehouse stops after N seconds idle (no compute charges). **Auto-resume**: starts on next query.
- **Multi-cluster warehouse**: horizontal scaling — `min/max` clusters to handle concurrency spikes.

In [0]:
# Illustrative Snowflake SQL (run inside a Snowflake worksheet, not Spark)
snowflake_sql = """
-- Create a warehouse with auto-suspend/resume and multi-cluster
CREATE WAREHOUSE IF NOT EXISTS etl_wh
  WAREHOUSE_SIZE = 'LARGE'
  AUTO_SUSPEND = 60          -- seconds idle -> suspend
  AUTO_RESUME  = TRUE
  INITIALLY_SUSPENDED = TRUE
  MIN_CLUSTER_COUNT = 1
  MAX_CLUSTER_COUNT = 3;     -- horizontal scaling for concurrency

-- Sizes (credit/hour roughly doubles per step):
-- X-Small, Small, Medium, Large, X-Large, 2X-Large, 3X-Large, 4X-Large, 5X-Large, 6X-Large

-- Inspect warehouses
SHOW WAREHOUSES;
"""
print(snowflake_sql)

print("Architecture: Storage (object store) + Cloud Services (metadata) + Virtual Warehouses (compute).")
print("Compute & storage scale independently -> pay-per-use.")

## Q15: Snowflake — Stages, File Formats & `COPY INTO` [INTERMEDIATE]

### Problem
Load CSV files from an S3 bucket into a Snowflake table. Walk through: named stage, file format, validation (`VALIDATE`), then `COPY INTO` with `ON_ERROR`. Explain internal vs external stages and why `COPY INTO` is idempotent per file (loads each file once, tracked in `LOAD_HISTORY`).

### Key Concept
- **Stage**: a reference to where data files live — internal (Snowflake-managed) or external (S3/GCS/Azure via STORAGE INTEGRATION).
- **File format**: defines CSV/JSON/Parquet parsing (delimiter, skip header, null markers).
- **`COPY INTO ... FROM @stage`**: bulk load; Snowflake tracks loaded files so re-running won't duplicate.
- `ON_ERROR = 'CONTINUE' | 'SKIP_FILE' | 'ABORT_STATEMENT'` controls bad-row handling.
- `VALIDATE TABLE ...` inspects load errors after the fact.

In [0]:
# Illustrative Snowflake SQL (run in a Snowflake worksheet)
snowflake_sql = """
-- 1. Storage integration + external stage (S3)
CREATE STORAGE INTEGRATION s3_int
  TYPE = AWS_S3
  STORAGE_PROVIDER = 'S3'
  STORAGE_AWS_ROLE_ARN = 'arn:aws:iam::123456789012:role/snowflake'
  STORAGE_ALLOWED_LOCATIONS = ('s3://my-bucket/orders/');

CREATE OR REPLACE STAGE orders_stage
  STORAGE_INTEGRATION = s3_int
  URL = 's3://my-bucket/orders/'
  FILE_FORMAT = (TYPE = CSV FIELD_DELIMITER=',' SKIP_HEADER=1 NULL_IF=('','NULL') EMPTY_FIELD_AS_NULL=TRUE);

-- 2. Target table
CREATE TABLE IF NOT EXISTS orders_raw (
  order_id NUMBER, customer_id NUMBER, amount NUMBER(10,2), order_date DATE
);

-- 3. Preview / validate before loading
LIST @orders_stage;
VALIDATE TABLE orders_raw, JOB_ID => NULL;  -- after a load to see errors

-- 4. Idempotent bulk load — Snowflake skips already-loaded files
COPY INTO orders_raw
  FROM @orders_stage
  ON_ERROR = 'SKIP_FILE'
  FORCE = FALSE;          -- FALSE = don't reload already-loaded files (default)

-- 5. Check what was loaded
SELECT * FROM SNOWFLAKE.ACCOUNT_USAGE.LOAD_HISTORY WHERE table_name='ORDERS_RAW';
"""
print(snowflake_sql)
print("\nStages: internal (Snowflake-managed) vs external (S3/GCS/Azure via storage integration).")
print("COPY INTO is idempotent per file via LOAD_HISTORY tracking.")

## Q16: Monitoring, Logging & Orchestration — Databricks Jobs Task Types [BASIC]

### Problem
Name the main **task types** in a Databricks Job and when to use each. Explain how tasks are wired into a **DAG** with dependencies.

### Key Concept
| Task type | Use for |
|---|---|
| Notebook | PySpark/Python logic in a notebook |
| Python script | Reusable `.py` modules / parameterized scripts |
| SQL | Run a SQL query or file (warehouses) |
| Pipeline (SDP) | Run a Lakeflow Spark Declarative Pipeline |
| DLT/condition | `Run If` branch task (conditional execution based on prior task status) |
| Run job task | Trigger another job (parent/child composition) |
| dbt | Run dbt transformations |

Dependencies: each task has **Depends on** (upstream tasks) and a **Run If** condition (`All done`, `All succeeded`, `At least one succeeded`, `None failed`). Retries + retry-on-failure are configurable per task.

In [0]:
# Conceptual outline of a multi-task job DAG (JSON shape, not executed here)
import json

job_spec = {
  "name": "orders_pipeline",
  "tasks": [
    {"task_key": "extract",  "notebook_task": {"notebook_path": "/etl/extract"}, "max_retries": 2},
    {"task_key": "transform","notebook_task": {"notebook_path": "/etl/transform"}, "depends_on": [{"task_key": "extract", "outcome": "success"}]},
    {"task_key": "quality",  "sql_task":      {"file": {"path": "/etl/checks.sql"}}, "depends_on": [{"task_key": "transform"}]},
    {"task_key": "load",     "pipeline_task":{"pipeline_id": "abc"},               "depends_on": [{"task_key": "quality", "outcome": "success"}]},
    {"task_key": "alert",    "condition_task":{"op": "EQUAL_TO","left": "{{job.status}}","right": "FAILED"}, "depends_on": [{"task_key": "load", "outcome": "all_done"}]},
  ],
}
print(json.dumps(job_spec, indent=2))
print("\nRun If options: All done | All succeeded | At least one succeeded | None failed.")
print("Task types: notebook, python_script, sql, pipeline, condition (Run If), run_job, dbt.")

## Q17: Monitoring — Job Triggers: Cron vs File Arrival vs Table Update [INTERMEDIATE]

### Problem
Compare the three Databricks Jobs trigger types. When would you choose each? Explain the **file arrival** trigger (event-driven on new files in a UC volume/storage) and the **table update** trigger (fires when a watched Delta table changes), and how they differ from a fixed **cron** schedule.

### Key Concept
| Trigger | Fires when | Best for |
|---|---|---|
| **Scheduled (cron)** | Fixed time cadence | Batch jobs, periodic reports |
| **File arrival** | New file appears in a watched storage/volume location | Event-driven ingestion after upstream drops files |
| **Table update** | A watched Delta table is updated (INSERT/UPDATE/DELETE) | Chained pipelines — downstream runs after upstream Delta changes |

- File arrival & table update are **event-driven** (low latency, no polling) vs cron (time-driven, possible idle runs).
- Table update trigger avoids both over- and under-scheduling dependent jobs.

In [0]:
import json

triggers = {
  "cron": {
    "quartz_cron_expression": "0 0 9 * * ?",   # daily 9 AM
    "timezone_id": "Asia/Kolkata",
  },
  "file_arrival": {
    "file_arrival": {
      "url": "s3://my-bucket/landing/orders/",
      "min_time_between_triggers_seconds": 120,
      "wait_after_last_change_seconds": 60,
    },
  },
  "table_update": {
    "table_update": {
      "table_name": "main.staging.orders_raw",
      "condition": "new_rows > 0",
    },
  },
}
for name, spec in triggers.items():
    print(f"--- {name} ---")
    print(json.dumps(spec, indent=2))

print("\nWhen to pick: cron = fixed cadence; file_arrival = upstream drops files;")
print("table_update = downstream should run when an upstream Delta table changes.")

## Q18: Monitoring — Delta `DESCRIBE HISTORY` & Time Travel for Debugging/Rollback [INTERMEDIATE]

### Problem
At 3 AM a job overwrote the wrong partition in `orders` Delta table. Use `DESCRIBE HISTORY` to find the bad commit and **time-travel restore** the table to its pre-failure state (`RESTORE TABLE ... VERSION AS OF`). Also explain `TIMESTAMP AS OF` and why Delta enables cheap rollback without re-ingestion.

### Key Concept
- `DESCRIBE HISTORY table` lists all commits (version, operation, timestamp, operation metrics).
- `SELECT * FROM table VERSION AS OF <v>` / `TIMESTAMP AS OF <ts>` reads a past snapshot.
- `RESTORE TABLE ... TO VERSION AS OF <v>` rewinds the table to that version (creates a new commit; no data copy).
- Time travel retention: `delta.logRetentionDuration` (default 30 days), `delta.deletedFileRetentionDuration` (default 7 days).

In [0]:
from pyspark.sql import Row

spark.sql("DROP TABLE IF EXISTS tmp_orders_hist_07")
spark.sql("CREATE TABLE tmp_orders_hist_07 (id INT, status STRING) USING DELTA")
spark.sql("INSERT INTO tmp_orders_hist_07 VALUES (1, 'NEW'), (2, 'NEW')")
spark.sql("INSERT INTO tmp_orders_hist_07 VALUES (3, 'NEW')")
spark.sql("UPDATE tmp_orders_hist_07 SET status='BAD' WHERE id=1")  # simulate bad write

# 1. Inspect history to find the bad commit
spark.sql("DESCRIBE HISTORY tmp_orders_hist_07").select("version", "operation", "operationParameters").show(truncate=False)

# 2. Time-travel read of version 1 (before the bad UPDATE)
spark.sql("SELECT * FROM tmp_orders_hist_07 VERSION AS OF 1 ORDER BY id").show()

# 3. Restore to the pre-failure version
spark.sql("RESTORE TABLE tmp_orders_hist_07 TO VERSION AS OF 1")
spark.sql("SELECT * FROM tmp_orders_hist_07 ORDER BY id").show()

# 4. Show retention settings (config)
spark.sql("DESCRIBE HISTORY tmp_orders_hist_07").select("version", "operation").show()
print("delta.logRetentionDuration default = 30 days; deletedFileRetentionDuration default = 7 days.")
print("RESTORE creates a new commit — cheap, no re-ingestion.")

## Day 07 — Score Card

Tick each question after solving it self-timed, then grade yourself.

| # | Category | Question | Difficulty | Solved? | Time (min) | Confidence (1-5) |
|---|----------|---------|------------|---------|------------|------------------|
| 1 | SQL Advanced | NTILE quartiles | BASIC | [ ] |  |  |
| 2 | SQL Advanced | Rolling 7-day MA (RANGE) | INTERMEDIATE | [ ] |  |  |
| 3 | Python | Two Sum | BASIC | [ ] |  |  |
| 4 | Python | Next Greater Element (stack) | INTERMEDIATE | [ ] |  |  |
| 5 | PySpark | when/otherwise | BASIC | [ ] |  |  |
| 6 | PySpark | collect_set / collect_list | INTERMEDIATE | [ ] |  |  |
| 7 | Spark Opt | repartition vs coalesce | BASIC | [ ] |  |  |
| 8 | Spark Opt | shuffle.partitions + AQE | INTERMEDIATE | [ ] |  |  |
| 9 | ETL | Delta MERGE / SCD1 | BASIC | [ ] |  |  |
| 10 | ETL | Auto Loader schema evolution | INTERMEDIATE | [ ] |  |  |
| 11 | ETL | foreachBatch idempotent sink | INTERMEDIATE | [ ] |  |  |
| 12 | Data Modeling | Kimball vs Inmon | BASIC | [ ] |  |  |
| 13 | Data Modeling | Conformed / role-playing dims | INTERMEDIATE | [ ] |  |  |
| 14 | Snowflake | Architecture & warehouses | BASIC | [ ] |  |  |
| 15 | Snowflake | Stages & COPY INTO | INTERMEDIATE | [ ] |  |  |
| 16 | Monitoring | Jobs task types | BASIC | [ ] |  |  |
| 17 | Monitoring | Job triggers | INTERMEDIATE | [ ] |  |  |
| 18 | Monitoring | Delta history & time travel | INTERMEDIATE | [ ] |  |  |

**Totals**: 18 questions | 7 BASIC | 11 INTERMEDIATE | Target time ~90 min

### Self-assessment
- **Areas strong**: 
- **Areas to revise**: 
- **Tomorrow (Day 08)**: step up to INTERMEDIATE → ADVANCED (Week 2 begins).